# 🛡️ SAFIR — Adim Adim Calisma (Jupyter Walkthrough)

**Saha Analiz ve Farkindalik Icin Yapay Zeka Destekli Karar Sistemi** pipeline'ini asama asama calistirir; her adimin girdi/ciktisi ekranda gorunur. Video **Gemini** ile analiz edilir.

Akis: **CPU Sampler → Temsili Kareler → VLM → Olay Tespiti → RAG → Ajan → Otomatik Eskalasyon → Rapor**

## 0) Kurulum ve Ayarlar
- `USE_MOCK=False` → `config.yaml`'daki backend (**Gemini**; `GEMINI_API_KEY` gerekli).
- `USE_FAKE_RAG=True` → agir embedding modelini (bge-m3 ~2GB) indirmez; hafif mevzuat kullanir (demo hizli).

In [ ]:
import sys, os
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = _here if (_here / 'src').exists() else _here.parent
assert (PROJECT_ROOT / 'src').exists(), 'Proje koku (safir-ai/) bulunamadi.'
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# ---- AYARLAR ----
USE_MOCK = False       # False: Gemini | True: offline
USE_FAKE_RAG = True    # True: bge-m3 indirmez (hizli) | False: gercek FAISS RAG

from src.utils.config_loader import load_config
config = load_config()
if USE_MOCK:
    config = config.model_copy(update={'app': config.app.model_copy(
        update={'use_mock_vlm': True, 'use_mock_llm': True})})
print('VLM backend :', config.vlm.active_model, '(', config.vlm.models[config.vlm.active_model].model_name, ')')
print('LLM backend :', config.llm.active_model)
print('MOCK        :', USE_MOCK, '| FAKE_RAG:', USE_FAKE_RAG)
if not USE_MOCK and not os.environ.get('GEMINI_API_KEY'):
    print('\n[UYARI] GEMINI_API_KEY tanimli degil. Terminalde:  set GEMINI_API_KEY=...  (veya notebook baslatmadan once export/$env).')

## 📥 Video Girisi
Asagidaki **kutudan bir video sec** (surukle-birak veya tikla) ya da bir dosya yolu yaz. Hicbiri verilmezse kucuk bir **sentetik** video otomatik uretilir.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept='video/*', multiple=False, description='📹 Video sec')
path_box = widgets.Text(value='', placeholder='veya yol: data/ornek.mp4', description='Yol:')
display(widgets.VBox([uploader, path_box]))
print('Video secip ALT hucreyi calistir. (Sentetik istersen ikisini de bos birak.)')

In [ ]:
import tempfile, cv2, numpy as np

def _resolve_video():
    # 1) Yuklenen dosya (ipywidgets 8: tuple; 7: dict)
    val = uploader.value
    if val:
        item = (list(val.values())[0] if isinstance(val, dict) else val[0])
        name = item.get('name') or (item.get('metadata', {}) or {}).get('name') or 'uploaded.mp4'
        content = item['content']
        Path('data').mkdir(exist_ok=True)
        p = f'data/{name}'
        Path(p).write_bytes(bytes(content))
        return p
    # 2) Elle girilen yol
    if path_box.value and Path(path_box.value).exists():
        return path_box.value
    # 3) Sentetik fallback
    tmp = tempfile.mkdtemp(); p = str(Path(tmp) / 'synthetic.mp4')
    frames = [np.full((120, 160, 3), 30, np.uint8) for _ in range(60)]
    for i in range(20, 38):
        cv2.rectangle(frames[i], (20, 20), (140, 100), (210, 210, 210), -1)
    w = cv2.VideoWriter(p, cv2.VideoWriter_fourcc(*'mp4v'), 25.0, (160, 120))
    for f in frames: w.write(f)
    w.release()
    print('(Sentetik video uretildi.)')
    return p

VIDEO_PATH = _resolve_video()
print('Kullanilan video:', VIDEO_PATH)

## 1) Adaptive Frame Sampler (CPU)
Video CPU'da taranir; yalnizca **kanit kareleri** secilir (gurultu tabani dusulmus degisim). VLM'e sadece bunlar gider — GPU tasarrufu buradan.

In [ ]:
from src.sampler.adaptive_sampler import sampler_from_config

sampler = sampler_from_config(config.sampler)
evidence = sampler.process_video(VIDEO_PATH, sample_fps=config.sampler.sample_fps)
s = sampler.last_run_stats
print(f'Kanit karesi     : {len(evidence)}')
print(f'Taranan ham kare : {s.total_frames_scanned}')
print(f'Elenen kare      : {s.eliminated_frame_count}  (GPU tasarrufu %{s.eliminated_ratio_pct})')
print(f'Sure             : {s.elapsed_sec}s')

## 2) Olay Kumeleme + VLM'e Gidecek Kareler (pre / peak / post)
Kanit kareleri **Olay Gruplari**na kumelenir; her grup icin zirvenin oncesi/sonrasi da eklenir. **Asagida VLM'e GONDERILECEK kareler tam olarak gorunur** — model bunlari bir dizi olarak yorumlayacak.

In [ ]:
import base64
from IPython.display import Image as IPyImage, display
from src.sampler.context.representative_frame_extractor import RepresentativeFrameExtractor

clusters = sampler.cluster_events(evidence)
extractor = RepresentativeFrameExtractor(
    config.sampler.pre_peak_offset_sec, config.sampler.post_peak_offset_sec)
for c in clusters:
    c.representative_frames = extractor.extract(VIDEO_PATH, c.peak_frame)

print(f'{len(clusters)} olay grubu\n')
for c in clusters:
    print(f'=== Olay #{c.event_id} — VLM\'e gidecek {len(c.representative_frames)} kare ===')
    imgs = []
    for rf in c.representative_frames:
        _, b64 = rf.base64_image.split(',', 1)
        imgs.append(IPyImage(data=base64.b64decode(b64), width=200))
        print(f'  • {rf.label:<11} @ {rf.timestamp_str}')
    display(widgets.HBox([widgets.Image(value=i.data, format='jpeg', width=200) for i in imgs]))

## 3) Hibrit Bellek / RAG servisi
Mevzuat aramasini yapan servis. `USE_FAKE_RAG=True` iken hafif sahte mevzuat kullanilir (agir model indirilmez).

In [ ]:
if USE_FAKE_RAG:
    from dataclasses import dataclass
    @dataclass
    class _Doc:
        text: str
        score: float = 1.0
    class _FakeRAG:
        def seed_default_regulations(self): pass
        def query(self, q, top_k=None):
            return [
                _Doc('ISG Yonetmeligi Madde 24: KKD (baret/yelek) zorunludur.'),
                _Doc('Operasyonel Kural OK-07: Forklift trafiginde yaya gecitleri acik tutulmalidir.'),
            ][:(top_k or 2)]
    rag_service = _FakeRAG()
else:
    from src.memory.embedding_rag_service import EmbeddingRAGService
    rag_service = EmbeddingRAGService(config.memory.embedding, config.memory.faiss)
    rag_service.seed_default_regulations()
print('RAG servisi hazir:', type(rag_service).__name__)

## 4) VLM — Gorsel Anlama (Gemini)
Yukaridaki kareler VLM'e gonderilir. Iki cikti: **insan-okur** Turkce gozlem ve **makine-okur** yapilandirilmis olaylar (`EVENTS_JSON`: tip/zaman/guven).

In [ ]:
from src.vlm.factory import get_vlm_client

vlm = get_vlm_client(config.vlm, use_mock=config.app.use_mock_vlm)
vlm_response = vlm.describe_events(clusters, prompt='Sahnede riskli bir durum var mi degerlendir.')

print('Model:', vlm_response.model_name, f'({vlm_response.latency_ms:.0f} ms)')
print('\n----- VLM Gozlemi -----\n')
print(vlm_response.description)
print('\n----- Yapilandirilmis olaylar (EVENTS_JSON) -----')
for e in vlm_response.structured_events:
    print('  ', e)

## 5) Olay Analizi — Neye Karar Verildi?
VLM ciktisi tipli olaylara cevrilir (**once** model-tabanli EVENTS_JSON, yoksa anahtar-kelime yedegi), sonra zamansal iliskilendirme ve **kural motoru** ile tetiklenen ISG kurallari bulunur.

In [ ]:
from src.event_analysis.event_engine import EventEngine
from src.event_analysis.temporal_reasoner import TemporalReasoner, DEFAULT_RELATION_WINDOW_SEC
from src.event_analysis.rule_engine import RuleEngine
from src.event_analysis.schemas import EventEngineInput
from src.agent.tools import RetrieverTool

engine_input = EventEngineInput.from_vlm_response(vlm_response, timestamp=clusters[-1].end_time)
detected = EventEngine().detect(engine_input)
print('1) Tespit edilen olaylar (DetectedEvent):')
for d in detected:
    print(f'   - {d.event_type:<24} guven={d.confidence:.2f}  keywords={d.matched_keywords}')

temporal = TemporalReasoner(relation_window_sec=DEFAULT_RELATION_WINDOW_SEC).reason(detected)
print('\n2) Zamansal olaylar (TemporalEvent):')
for t in temporal:
    print(f'   - {t.event_type:<24} tekrar={t.occurrence_count} sure={t.duration:.1f}s')

rules = RuleEngine(retriever=RetrieverTool(rag_service)).evaluate(temporal)
print('\n3) Tetiklenen ISG kurallari (RuleMatch):')
for r in rules:
    print(f'   - [{r.rule_id}] ({r.severity}) {r.rule_description}')

## 6) LangGraph Ajani — Muhakeme ve Karar
Ajan; gozlemi, mevzuati ve olay sinyallerini alir, gerekirse araclarini (sql / retriever / timeline / verification) cagirir ve **sartname-uyumlu JSON** karar uretir: risk skoru, seviye, ozet, aksiyonlar.

In [ ]:
from src.agent.langgraph_agent import SafirAgent

agent = SafirAgent(
    llm_config=config.llm, agent_config=config.agent,
    event_store=None, rag_service=rag_service, use_mock_llm=config.app.use_mock_llm)

context = (
    f'## Guncel Gozlem\n{vlm_response.description}\n\n'
    '## Kullanici Istemi\nSahnede riskli bir durum var mi degerlendir.'
)
decision = agent.run(context)

print('Risk skoru :', decision.risk_score, f'({decision.risk_level})')
print('Ozet       :', decision.summary)
print('Aksiyonlar :')
for a in decision.actions:
    print('   -', a)

## 7) Otomatik Eskalasyon (Human-on-the-Loop)
Bloke edici operator onayi YOK: sistem risk'e gore kademeyi kendisi secer. Yuksek/kritikte saha alarmi **otomatik** tetiklenir; operator sonradan denetler.

In [ ]:
from src.decision.escalation import EscalationPolicy

esc = EscalationPolicy(config.escalation).evaluate(
    risk_score=decision.risk_score, risk_level=decision.risk_level,
    recommended_action=decision.recommended_action, summary=decision.summary)
print('Kademe         :', esc.tier.value)
print('Otomatik alarm :', esc.auto_dispatched)
print('alert_id       :', esc.alert_id)
print('Gerekce        :', esc.reason)

## 8) Uctan Uca Entegre Kosu — Nihai Rapor
Tum asamalar `SafirPipeline.run()` icinde birlesir; tek cagriyla nihai **sartname-uyumlu JSON** uretilir.

In [ ]:
import json
import src.main as safir_main

if USE_FAKE_RAG:
    safir_main.EmbeddingRAGService = lambda *a, **k: rag_service

pipeline = safir_main.SafirPipeline(config)
report = pipeline.run(VIDEO_PATH, 'Sahnede riskli bir durum var mi degerlendir.')

print('risk          :', report.risk_score, f'({report.risk_level})')
print('eskalasyon    :', report.escalation_tier, '| otomatik:', report.auto_dispatched)
print('tespit tipler :', report.detected_event_types)
print('\n===== SARTNAME UYUMLU JSON =====')
print(json.dumps(report.to_sartname_json(), ensure_ascii=False, indent=2))

---
Bu defter SAFIR'in tam akisini adim adim gosterdi. Operator paneli ayni backend'i kullanir: `streamlit run src/ui/dashboard.py`.